# Foundational SAT Skill Gap Prediction

**Synthetic data only.** This notebook does not use real student information.

## Motivation
SAT tutoring platforms track mastery across dozens of Reading/Writing skills. Students are rarely tested on every skill each month—often only 30–40 of ~72 skills have scores. This prototype uses **linear algebra** to predict likely weaknesses on **untested foundational skills** from partial performance data.

## Linear algebra concepts used
- **Student-skill matrix** \(S \in \mathbb{R}^{n \times m}\)
- **Student vectors** (rows of \(S\)) in skill space
- **Dot products** and **vector norms**
- **Cosine similarity** between student vectors
- **Weighted imputation** of missing matrix entries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lib import (
    N_STUDENTS, N_SKILLS, MASK_FRACTION, RANDOM_SEED,
    run_pipeline, analyze_student, cosine_similarity_observed,
    save_figures, skill_metadata_df,
)
from export_frontend import export_dashboard

sns.set_theme(style="whitegrid")

## 1. Skill catalog (72 SAT skills)

In [ ]:
meta = skill_metadata_df()
meta.head(10)

## 2. Student-skill matrix

Each student is a vector \(\mathbf{s}_i \in \mathbb{R}^{72}\) of mastery percentages. Stacking rows gives matrix \(S\):

$$S = \begin{bmatrix} \mathbf{s}_1 \\ \mathbf{s}_2 \\ \vdots \\ \mathbf{s}_n \end{bmatrix}$$

In [ ]:
result = run_pipeline(seed=RANDOM_SEED)
S = result.scores
M = result.mask
print(f"Matrix shape: {S.shape}  (students × skills)")
print(f"Mask fraction hidden: {(~M).mean():.1%}")

## 3. Masking untested skills

About 40% of entries are hidden per student to simulate skills not yet assessed.

In [ ]:
S_masked = S.copy()
S_masked[~M] = np.nan
pd.DataFrame(S_masked[:5, :8], index=result.student_ids[:5], columns=result.skill_ids[:8])

## 4. Linear algebra: dot product, norm, cosine similarity

**Dot product:** \(\mathbf{u} \cdot \mathbf{v} = \sum_k u_k v_k\)

**Norm:** \(\|\mathbf{u}\| = \sqrt{\mathbf{u} \cdot \mathbf{u}}\)

**Cosine similarity** (on shared observed indices \(\Omega\)):

$$\text{sim}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u}_\Omega \cdot \mathbf{v}_\Omega}{\|\mathbf{u}_\Omega\| \|\mathbf{v}_\Omega\|}$$

This measures the angle between student performance vectors—students pointing the same direction in skill space are similar.

In [ ]:
target_idx = result.default_student_idx
peer_idx = 1
shared = M[target_idx] & M[peer_idx]
u = S[target_idx, shared]
v = S[peer_idx, shared]
dot = np.dot(u, v)
norm_u = np.linalg.norm(u)
norm_v = np.linalg.norm(v)
cos_sim = dot / (norm_u * norm_v)
print(f"Manual dot product: {dot:.2f}")
print(f"||u||={norm_u:.2f}, ||v||={norm_v:.2f}")
print(f"Cosine similarity: {cos_sim:.4f}")
print(f"Function check: {cosine_similarity_observed(S[target_idx], S[peer_idx], shared):.4f}")

## 5. Predict missing scores & rank foundational gaps

**Prediction:** weighted average of similar peers who were tested on skill \(j\):

$$\hat{s}_j = \frac{\sum_i \text{sim}(u, v_i) \cdot v_{i,j}}{\sum_i \text{sim}(u, v_i)}$$

**Priority:** \(\text{priority}_j = (100 - \hat{s}_j) \times w_j\) where \(w_j\) is foundational weight.

In [ ]:
analysis = analyze_student(result, target_idx)
sid = result.student_ids[target_idx]
recs = pd.DataFrame(analysis["recommendations"])
print(f"Target: {sid}")
print(analysis["summary"]["interpretation"])
recs.head(10)[["rank", "skill_name", "category", "level", "predicted_mastery", "foundational_weight", "priority_score", "reason"]]

## 6. Visualizations

In [ ]:
save_figures(result, target_idx, analysis)
from IPython.display import Image, display
display(Image("output/figures/heatmap.png"))
display(Image("output/figures/top_recommendations.png"))
display(Image("output/figures/nearest_peers.png"))

## 7. Export for dashboard

In [ ]:
export_dashboard()
print("Exported frontend/public/data/dashboard.json")

## Assignment requirements

1. **Non-trivial linear algebra:** Students are modeled as vectors; performance is a matrix; cosine similarity uses dot products and norms; predictions use weighted vector comparisons.
2. **Real problem:** Platforms cannot test every SAT skill monthly—this model surfaces hidden foundational weaknesses to prioritize next practice.
3. **Verifiable artifacts:** Code, synthetic CSVs, visualizations, recommendation tables, dashboard JSON, and this written explanation.